In [4]:
import pandas as pd
import re
from langdetect import detect

def bersihkan_dataset(file_input, file_output, provider_name):
    print(f"\n=======================================================")
    print(f"MEMULAI PROSES UNTUK: {provider_name.upper()} ({file_input})")
    print(f"=======================================================")
    
    # 1. BACA DATA
    print("1... Membaca data CSV...")
    df = pd.read_csv(file_input)

    # 2. MENGGABUNGKAN KOLOM & MENGAMBIL USERNAME
    print("2... Mengekstrak Username dan Teks dari kolom-kolom acak...")
    def gabungkan_teks_baris(row):
        teks_kumpul = []
        # Daftar teks bawaan UI Twitter/X yang ingin diabaikan
        frasa_ui_twitter = ['replying to', 'show original', 'automated by', 'parody account', 'quote']
        
        for nilai in row:
            teks_str = str(nilai).strip()
            teks_lower = teks_str.lower()
            
            if teks_lower == 'nan' or teks_str == '' or teks_str.isnumeric():
                continue
            
            # Abaikan URL gambar/link, username (@), dan teks yang terlalu pendek
            if not teks_str.startswith('http') and not teks_str.startswith('@') and len(teks_str) > 3:
                # Lewati jika cell ini merupakan UI Twitter (misal: "Translated from English", "Replying to", dll)
                if any(teks_lower.startswith(frasa) for frasa in frasa_ui_twitter) or teks_lower.startswith('translated from'):
                    continue
                
                teks_kumpul.append(teks_str)
                
        return ' '.join(teks_kumpul)

    df['Teks_Mentah'] = df.apply(gabungkan_teks_baris, axis=1)

    def dapatkan_username(row):
        for nilai in row:
            teks_str = str(nilai).strip()
            if teks_str.startswith('@') and len(teks_str) > 1:
                return teks_str
            # Support format "Username@Handle", jadi kita coba cari @ di tengah string juga
            elif '@' in teks_str and len(teks_str.split('@')[1]) > 1:
                return '@' + teks_str.split('@')[1]
        return 'Tanpa_Username'

    df['Username'] = df.apply(dapatkan_username, axis=1)

    # 3. PEMBERSIHAN TEKS
    print("3... Membersihkan teks (Cleaning)...")
    def bersihkan_teks(text):
        text = text.lower() 
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) 
        text = re.sub(r'\S+\.com\S*', '', text) # Menghapus sisa link berformat .com
        text = re.sub(r'\@\w+|\#', '', text) 
        
        # Ekstra pembersihan frasa UI Twitter yang mungkin lolos
        text = re.sub(r'\breplying to\b', ' ', text)
        text = re.sub(r'\bshow original\b', ' ', text)
        text = re.sub(r'\btranslated from \w+\b', ' ', text)
        text = re.sub(r'\bautomated by\b', ' ', text)
        text = re.sub(r'\bparody account\b', ' ', text)
        text = re.sub(r'\bquote\b', ' ', text)
        
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\bcom\b', ' ', text) # Menghapus sisa kata 'com' jika masih ada
        text = re.sub(r'\s+', ' ', text).strip() 
        return text

    df['Teks_Bersih'] = df['Teks_Mentah'].apply(bersihkan_teks)

    # 4. FILTER BAHASA
    print("4... Menyaring bahasa (Hanya Bahasa Indonesia)...")
    def filter_bahasa(text):
        try:
            if len(text) < 10: 
                return 'unknown'
            return detect(text)
        except:
            return 'unknown'

    df['Bahasa'] = df['Teks_Bersih'].apply(filter_bahasa)
    df_indo = df[df['Bahasa'] == 'id'].copy()

    # 5. MENYIAPKAN HASIL AKHIR (Tanpa mengubah Bahasa Gaul)
    print("5... Menggunakan teks asli tanpa mengubah bahasa gaul...")
    df_indo['Teks_Final'] = df_indo['Teks_Bersih']

    # 6. SIMPAN HASIL DAN HAPUS DUPLIKAT
    print("6... Melakukan pembersihan akhir (Hapus Akun Official, Tweet Iklan, & Duplikat)...")
    df_final = df_indo[['Username', 'Teks_Final']].rename(columns={'Teks_Final': 'Tweet'})
    df_final = df_final[df_final['Tweet'] != ''] 

    # Hapus data yang username-nya merupakan akun official (mengandung xl, indosat, atau telkomsel)
    df_final = df_final[~df_final['Username'].str.lower().str.contains('xl|indosat|telkomsel', na=False)]

    # >>> [NEW] Filter Khusus XL: Menghapus keyword iklan baju/celana yang nyasar sebagai ukuran "XL" <<<
    if provider_name.lower() == 'xl':
        # Penambahan filter keyword iklan merch (jersey, sepatu, dll) dan kata khusus (adidas, home, kucnicaway dll)
        kata_pakaian = 'baju|kaos|ukuran|size|celana|jaket|kemeja|lingkar dada|ld|hoodie|sweater|kardigan|dress|rok|gamis|atasan|bawahan|thrift|preloved|outfit|kerah|motif|jersey|adidas|away|home|kucnicaway|ready stock|xxl|titipjual|merch|sepatu|apparel'
        jml_sebelum_pakaian = len(df_final)
        df_final = df_final[~df_final['Tweet'].str.lower().str.contains(kata_pakaian, na=False)]
        print(f"    -> Dihapus {jml_sebelum_pakaian - len(df_final)} data terkait ukuran/pakaian/merchandise (Noise XL).")

    # Hapus data duplikat (berdasarkan kolom Tweet)
    jumlah_sebelum = len(df_final)
    df_final = df_final.drop_duplicates(subset=['Tweet'])
    jumlah_setelah = len(df_final)
    print(f"    -> {jumlah_sebelum - jumlah_setelah} data duplikat berhasil dihapus.")

    df_final.to_csv(file_output, index=False)
    
    print(f"\nSELESAI! Data {provider_name} berhasil disimpan di: {file_output}")
    print(f"Total data bersih: {len(df_final)} baris\n")
    return df_final


# =======================================================
# JALANKAN UNTUK SEMUA DATA
# =======================================================
df_xl = bersihkan_dataset(
    file_input='../data/raw/data_xl.csv', 
    file_output='../data/processed/data_xl_bersih.csv',
    provider_name='XL'
)

df_telkomsel = bersihkan_dataset(
    file_input='../data/raw/data_telkomsel.csv', 
    file_output='../data/processed/data_telkomsel_bersih.csv',
    provider_name='Telkomsel'
)

df_indosat = bersihkan_dataset(
    file_input='../data/raw/data_indosat.csv', 
    file_output='../data/processed/data_indosat_bersih.csv',
    provider_name='Indosat'
)

print("--- Cuplikan Data XL Bersih ---")
print(df_xl.head(), "\n")

print("--- Cuplikan Data Telkomsel Bersih ---")
print(df_telkomsel.head(), "\n")

print("--- Cuplikan Data Indosat Bersih ---")
print(df_indosat.head())


MEMULAI PROSES UNTUK: XL (../data/raw/data_xl.csv)
1... Membaca data CSV...
2... Mengekstrak Username dan Teks dari kolom-kolom acak...
3... Membersihkan teks (Cleaning)...
4... Menyaring bahasa (Hanya Bahasa Indonesia)...
5... Menggunakan teks asli tanpa mengubah bahasa gaul...
6... Melakukan pembersihan akhir (Hapus Akun Official, Tweet Iklan, & Duplikat)...
    -> Dihapus 10 data terkait ukuran/pakaian/merchandise (Noise XL).
    -> 5 data duplikat berhasil dihapus.

SELESAI! Data XL berhasil disimpan di: ../data/processed/data_xl_bersih.csv
Total data bersih: 102 baris


MEMULAI PROSES UNTUK: TELKOMSEL (../data/raw/data_telkomsel.csv)
1... Membaca data CSV...
2... Mengekstrak Username dan Teks dari kolom-kolom acak...
3... Membersihkan teks (Cleaning)...
4... Menyaring bahasa (Hanya Bahasa Indonesia)...
5... Menggunakan teks asli tanpa mengubah bahasa gaul...
6... Melakukan pembersihan akhir (Hapus Akun Official, Tweet Iklan, & Duplikat)...
    -> 8 data duplikat berhasil dihapus.

In [5]:
import pandas as pd

print("=======================================================")
print("MEMULAI PROSES PENGGABUNGAN DATA (MERGE)")
print("=======================================================")

# 1. BACA DATA BERSIH
print("1... Membaca ketiga file data bersih...")
df_xl_bersih = pd.read_csv('../data/processed/data_xl_bersih.csv')
df_telkomsel_bersih = pd.read_csv('../data/processed/data_telkomsel_bersih.csv')
df_indosat_bersih = pd.read_csv('../data/processed/data_indosat_bersih.csv')

# 2. TAMBAHKAN LABEL PROVIDER (Agar kita tahu darimana asal tweet tersebut)
print("2... Menambahkan label asal provider untuk masing-masing dataset...")
df_xl_bersih['Provider'] = 'XL'
df_telkomsel_bersih['Provider'] = 'Telkomsel'
df_indosat_bersih['Provider'] = 'Indosat'

# 3. GABUNGKAN DATA
print("3... Menggabungkan ketiga data menjadi satu...")
df_gabungan = pd.concat([df_xl_bersih, df_telkomsel_bersih, df_indosat_bersih], ignore_index=True)

# 4. HAPUS DUPLIKAT SEKALI LAGI LINTAS PROVIDER (Opsional untuk berjaga-jaga jika ada user ngetweet sama ke 3 provider)
jumlah_sebelum_merge = len(df_gabungan)
df_gabungan = df_gabungan.drop_duplicates(subset=['Tweet'])
jumlah_setelah_merge = len(df_gabungan)
print(f"    -> Ditemukan dan dihapus {jumlah_sebelum_merge - jumlah_setelah_merge} data duplikat lintas provider.")

# 5. SIMPAN KE CSV
file_gabungan = '../data/processed/data_gabungan_bersih.csv'
df_gabungan.to_csv(file_gabungan, index=False)

print(f"\nSELESAI! Data gabungan berhasil disimpan di: {file_gabungan}")
print(f"Total baris data gabungan: {len(df_gabungan)} baris")

print("\n--- Cuplikan Data Gabungan (Acak) ---")
print(df_gabungan.sample(10))

MEMULAI PROSES PENGGABUNGAN DATA (MERGE)
1... Membaca ketiga file data bersih...
2... Menambahkan label asal provider untuk masing-masing dataset...
3... Menggabungkan ketiga data menjadi satu...
    -> Ditemukan dan dihapus 0 data duplikat lintas provider.

SELESAI! Data gabungan berhasil disimpan di: ../data/processed/data_gabungan_bersih.csv
Total baris data gabungan: 830 baris

--- Cuplikan Data Gabungan (Acak) ---
           Username                                              Tweet  \
265      @iamHadaJZ  betmensikalong apr sinyal telkomsel kenapa mal...   
188       @staexsyk  may sinyal telkomsel duhh ini kenapa sih gaada...   
411  @inside444mind  niye apr telkomsel jelek ini jaringan gara hab...   
462   @SRwirahadini             apr telkomsel jelek bgt ya jaringannya   
581      @m4ddieprz  maddie may entah ap normal gajelas tetep aja s...   
701      @keedehara  kirara indosat may yaampun gemas pandavva bima...   
363   @irezadigital  reza may telkomsel jelek sinyal ala ka

In [6]:
import pandas as pd
import os

def load_lexicon(file_path):
    lexicon = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            # Lewati baris kosong atau header
            if not line or line.startswith('word\tweight'):
                continue
            
            parts = line.split('\t')
            if len(parts) >= 2:
                word = parts[0].strip()
                try:
                    weight = int(parts[1].strip())
                    lexicon[word] = weight
                except ValueError:
                    pass
    return lexicon

# Path ke file lexicon
pos_lex_path = '../data/lexicon/positive_words.txt'
neg_lex_path = '../data/lexicon/negative_words.txt'

pos_lex = load_lexicon(pos_lex_path)
neg_lex = load_lexicon(neg_lex_path)
print(f"Berhasil memuat {len(pos_lex)} kata positif dan {len(neg_lex)} kata negatif.")

def calculate_sentiment(text):
    if pd.isna(text):
        return 0, 'neutral'
    
    words = str(text).split()
    score = 0
    for w in words:
        w_lower = w.lower()
        if w_lower in pos_lex:
            score += pos_lex[w_lower]
        elif w_lower in neg_lex:
            score += neg_lex[w_lower]
    
    if score > 0:
        sentiment = 'positive'
    elif score < 0:
        sentiment = 'negative'
    else:
        sentiment = 'neutral'
        
    return pd.Series([score, sentiment])

# Fungsi untuk memproses file
try:
    # Membaca data yang sudah dibersihkan
    df = pd.read_csv('../data/processed/data_gabungan_bersih.csv')
    
    # Menghitung skor dan sentimen
    print("Menghitung skor sentimen...")
    df[['Lexicon_Score', 'Sentiment']] = df['Tweet'].apply(calculate_sentiment)
    
    # Menyimpan data baru
    output_path = '../data/processed/data_gabungan_bersih_tersentimen.csv'
    df.to_csv(output_path, index=False)
    
    print(f"\nBerhasil menyimpan data baru di: {output_path}")
    print("\nDistribusi Sentimen:")
    print(df['Sentiment'].value_counts())
    
    display(df.head())
except Exception as e:
    print(f"Error: {e}")

Berhasil memuat 3609 kata positif dan 6607 kata negatif.
Menghitung skor sentimen...

Berhasil menyimpan data baru di: ../data/processed/data_gabungan_bersih_tersentimen.csv

Distribusi Sentimen:
Sentiment
negative    411
positive    329
neutral      90
Name: count, dtype: int64


,Username,Tweet,Provider,Lexicon_Score,Sentiment
0,@Jewwpeace,eyi fan account lagi eror kok sinyalku ilang i...,XL,-11,negative
1,@kueezya,kes owyul telkom e mkin menurun prah hampir e ...,XL,4,positive
2,@laboucheedilya,sekarang aku ga takut lagi buat scroll sosmed ...,XL,-7,negative
3,@feelshouten,nand speed test aku di kabupaten deli serdang ...,XL,7,positive
4,@vcdkz,kyumne jaringan klo bentuk nya manusia udh gua...,XL,9,positive
